# Anti-Goal Chess Benchmark @ NeurIPS 2026 — CHECK MODE (tiny, raises on failure)

Paired win/lose small-model chess study (see README). Results land in `results_check/` and are zipped for download.
- Positions + exact oracles: committed (`data/positions/`), generated once by `scripts/generate_positions.py`.
- Engine + dataset tests gate every run: `scripts/test_engine.py`.
- Sweep: `scripts/run_suite.py` (models x tasks x {{win,lose}}).

## 1. Get the repo (GitHub secret method)

The repo is **private**. On Kaggle, a secret reaches the notebook ONLY if it is **attached to this notebook** and the kernel is started AFTER attaching:

1. Notebook editor -> **+ Add** (top-right) -> **Add secret** -> select `GITHUB_TOKEN` (it must exist under Account settings -> Secrets; value = a classic PAT with `repo` scope).
2. **Save** the notebook (Ctrl+S).
3. **Kernel -> Restart & Run All** (env vars are injected at kernel start; plain "Run All" does NOT pick up newly attached secrets).

This cell reads the token from the env var, and falls back to Kaggle's own `kaggle_secrets` API if the env var is missing.

In [7]:
import os, subprocess, sys
from pathlib import Path

WORK = Path("/kaggle/working")
REPO = WORK / "neuro-symbolic-pathfinding"

def find_token():
    for name in ("GITHUB_TOKEN", "GH_TOKEN"):
        if os.environ.get(name):
            return os.environ[name]
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("GITHUB_TOKEN")
    except Exception:
        return None

if not REPO.exists():
    # diagnostic: what token-ish env vars are actually present?
    present = sorted(k for k in os.environ if "TOKEN" in k.upper() or "SECRET" in k.upper())
    print("token-ish env vars present:", present, flush=True)
    token = find_token()
    print("GITHUB_TOKEN resolved:", bool(token), flush=True)
    url = "https://github.com/Vedang-P/neuro-symbolic-pathfinding.git"
    if token:
        url = url.replace("https://", f"https://x-access-token:{token}@")
    res = subprocess.run(["git", "clone", "--quiet", url, str(REPO)],
                         capture_output=True, text=True)
    if res.returncode != 0:
        raise RuntimeError(
            "git clone failed. The token did not reach this run. Fix order: "
            "(1) + Add -> Add secret -> GITHUB_TOKEN; (2) SAVE the notebook; "
            "(3) Kernel -> Restart & Run All. Then check the diagnostic line above: "
            "if 'token-ish env vars present' is empty, the secret is not attached to "
            "THIS notebook. Stderr: " + res.stderr[-300:]
        )
os.chdir(REPO)
print("cwd:", Path.cwd())

cwd: /kaggle/working/neuro-symbolic-pathfinding


## 2. Dependencies

In [8]:
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-r", "requirements.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "uninstall", "--quiet", "-y", "wandb"], check=True)
import torch
print("torch", torch.__version__, "cuda", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

torch 2.10.0+cu128 cuda True Tesla T4


## 3. Stage runner (never raises; the verdict cell checks results)

In [9]:
import json, time, shutil
from pathlib import Path

STAGE_LOG = Path("results_check/stage_log.json")
def run_stage(name, args, timeout_min):
    Path("results_check").mkdir(parents=True, exist_ok=True)
    rec = {"stage": name, "status": "running", "elapsed_min": None}
    t0 = time.time()
    try:
        res = subprocess.run(args, timeout=timeout_min * 60)
        rec["status"] = "ok" if res.returncode == 0 else "failed"
        rec["returncode"] = res.returncode
    except subprocess.TimeoutExpired:
        rec["status"] = "timeout"
    except Exception as e:
        rec["status"] = "error"
        rec["error"] = str(e)[:200]
    rec["elapsed_min"] = round((time.time() - t0) / 60, 1)
    entries = json.loads(STAGE_LOG.read_text()) if STAGE_LOG.exists() else []
    entries.append(rec)
    STAGE_LOG.write_text(json.dumps(entries, indent=1))
    print(f"stage {name}: {rec['status']} ({rec['elapsed_min']}min)", flush=True)
    return rec["status"]

## 4. Gate: engine + dataset tests

In [10]:
status = run_stage("engine_tests", [sys.executable, "scripts/test_engine.py", "--quick"], 10)
if status != "ok":
    raise RuntimeError("engine tests failed -- see output above")

ok   sq_to_algebraic a1
ok   algebraic_to_sq h8
ok   algebraic_to_sq junk
ok   algebraic_to_sq short
ok   8x8 mate-in-1 Qg7-style
ok   3x3 KvK 2 legal moves
ok   value KvK draw
ok   value 3x3 KQvK win
ok   value 5x5 KQvK win
ok   value blocked pawn draw
ok   value bK can capture, draw
ok   value far pawn promotes, win
ok   value KQvKQ 3x3 draw
ok   win moves subset of legal
ok   lose moves subset of legal
ok   win and lose disjoint
ok   win position has win moves
ok   win position has non-win moves
ok   dataset cap-legal-8x8 non-empty
ok   cap-tewjc has oracle data
ok   cap-CIamA has oracle data
ok   cap-rSju2 has oracle data
ok   cap-JydUg has oracle data
ok   cap-KtyFN has oracle data
ok   cap-MKCXC has oracle data
ok   cap-FwvK6 has oracle data
ok   cap-IxL0T has oracle data
ok   cap-1nUO4 has oracle data
ok   cap-8NO1C has oracle data
ok   cap-6ByYN has oracle data
ok   cap-1jwCv has oracle data
ok   cap-UZc8R has oracle data
ok   cap-dCjBK has oracle data
ok   cap-qrCnH has oracle

## 5. Position generation sanity (tiny, exercises the oracle path)

In [11]:
status = run_stage("gen_check", [sys.executable, "scripts/generate_positions.py", "--check", "--out", "results_check/positions"], 10)
if status != "ok":
    raise RuntimeError("position generation check failed")

sm-3x3-win: 3 positions 0.1s
sm-3x3-draw: 3 positions 0.2s
sm-5x5-win: 3 positions 4.4s
sm-5x5-draw: 3 positions 18.8s
mate1-8x8: 3 positions 0.0s
mob-8x8: 3 positions 0.0s
total 23.5s -> results_check/positions
stage gen_check: ok (0.4min)


## 6. The chess sweep (models x tasks, paired win/lose)

In [12]:
sweep_args = [sys.executable, "scripts/run_suite.py", "--output_dir", "results_check/chess"]
if True:
    sweep_args.append("--check")
status = run_stage("chess_sweep", sweep_args, 25)
print("sweep:", status)

suite: 1 models x 10 task-variant cells (CHECK mode)

>>> smollm2-1.7b x cap-legal-8x8:grid (n=3)


Loading weights: 100%|██████████| 218/218 [00:01<00:00, 168.13it/s, Materializing param=model.norm.weight]                              


  [cap-legal-8x8 smollm2-1.7b grid] 3/3 (1.0s/position)
{
 "conditions": {
  "win": {
   "n": 3,
   "no_answer": 0,
   "parse_error": 0,
   "illegal": 3,
   "legal": 0,
   "compliant": 0,
   "noncompliant": 0,
   "undefined": 0,
   "compliance_of_legal": null,
   "parse_rate": 1.0,
   "legal_rate": 0.0,
   "compliance_strict": 0.0
  }
 }
}
    46s

>>> smollm2-1.7b x cap-legal-8x8:fen (n=3)


Loading weights: 100%|██████████| 218/218 [00:01<00:00, 176.16it/s, Materializing param=model.norm.weight]                              


  [cap-legal-8x8 smollm2-1.7b fen] 3/3 (0.8s/position)
{
 "conditions": {
  "win": {
   "n": 3,
   "no_answer": 0,
   "parse_error": 0,
   "illegal": 3,
   "legal": 0,
   "compliant": 0,
   "noncompliant": 0,
   "undefined": 0,
   "compliance_of_legal": null,
   "parse_rate": 1.0,
   "legal_rate": 0.0,
   "compliance_strict": 0.0
  }
 }
}
    16s

>>> smollm2-1.7b x mate1-lichess:grid (n=3)


...

  [sm-5x5-draw smollm2-1.7b grid] 3/3 (1.5s/position)
{
 "conditions": {
  "win": {
   "n": 3,
   "no_answer": 0,
   "parse_error": 0,
   "illegal": 3,
   "legal": 0,
   "compliant": 0,
   "noncompliant": 0,
   "undefined": 0,
   "compliance_of_legal": null,
   "parse_rate": 1.0,
   "legal_rate": 0.0,
   "compliance_strict": 0.0
  },
  "lose": {
   "n": 3,
   "no_answer": 0,
   "parse_error": 0,
   "illegal": 3,
   "legal": 0,
   "compliant": 0,
   "noncompliant": 0,
   "undefined": 0,
   "compliance_of_legal": null,
   "parse_rate": 1.0,
   "legal_rate": 0.0,
   "compliance_strict": 0.0
  }
 },
 "divergence_rate": null
}
    17s

>>> smollm2-1.7b x mate1-8x8:grid (n=3)


Loading weights: 100%|██████████| 218/218 [00:01<00:00, 179.20it/s, Materializing param=model.norm.weight]                              


  [mate1-8x8 smollm2-1.7b grid] 3/3 (1.6s/position)
{
 "conditions": {
  "win": {
   "n": 3,
   "no_answer": 0,
   "parse_error": 0,
   "illegal": 3,
   "legal": 0,
   "compliant": 0,
   "noncompliant": 0,
   "undefined": 0,
   "compliance_of_legal": null,
   "parse_rate": 1.0,
   "legal_rate": 0.0,
   "compliance_strict": 0.0
  },
  "lose": {
   "n": 3,
   "no_answer": 0,
   "parse_error": 0,
   "illegal": 3,
   "legal": 0,
   "compliant": 0,
   "noncompliant": 0,
   "undefined": 0,
   "compliance_of_legal": null,
   "parse_rate": 1.0,
   "legal_rate": 0.0,
   "compliance_strict": 0.0
  }
 },
 "divergence_rate": null
}
    18s

>>> smollm2-1.7b x mob-8x8:grid (n=3)


Loading weights: 100%|██████████| 218/218 [00:01<00:00, 176.69it/s, Materializing param=model.norm.weight]                              


  [mob-8x8 smollm2-1.7b grid] 3/3 (1.6s/position)
{
 "conditions": {
  "win": {
   "n": 3,
   "no_answer": 0,
   "parse_error": 0,
   "illegal": 3,
   "legal": 0,
   "compliant": 0,
   "noncompliant": 0,
   "undefined": 0,
   "compliance_of_legal": null,
   "parse_rate": 1.0,
   "legal_rate": 0.0,
   "compliance_strict": 0.0
  },
  "lose": {
   "n": 3,
   "no_answer": 0,
   "parse_error": 0,
   "illegal": 3,
   "legal": 0,
   "compliant": 0,
   "noncompliant": 0,
   "undefined": 0,
   "compliance_of_legal": null,
   "parse_rate": 1.0,
   "legal_rate": 0.0,
   "compliance_strict": 0.0
  }
 },
 "divergence_rate": null
}
    18s

comparison table: /kaggle/working/neuro-symbolic-pathfinding/results_check/chess/comparison_table.csv (18 rows) total 205s
stage chess_sweep: ok (3.4min)
sweep: ok


## 7. Results table

In [13]:
import pandas as pd
csv_path = Path("results_check/chess/comparison_table.csv")
if csv_path.exists():
    df = pd.read_csv(csv_path)
    display(df)
    print("rows:", len(df))
else:
    print("no comparison table -- sweep did not complete")

,model,task,variant,condition,n,parse_rate,legal_rate,compliance_of_legal,compliance_strict,undefined
0,smollm2-1.7b,cap-legal-8x8,grid,win,3,1.0,0.0,NaN,0.0,0
1,smollm2-1.7b,cap-legal-8x8,fen,win,3,1.0,0.0,NaN,0.0,0
2,smollm2-1.7b,mate1-lichess,grid,win,3,1.0,0.0,NaN,0.0,0
3,smollm2-1.7b,mate1-lichess,grid,lose,3,1.0,0.0,NaN,0.0,0
4,smollm2-1.7b,mate1-lichess,fen,win,3,1.0,0.0,NaN,0.0,0
5,smollm2-1.7b,mate1-lichess,fen,lose,3,1.0,0.0,NaN,0.0,0
6,smollm2-1.7b,sm-3x3-win,grid,win,3,1.0,0.0,NaN,0.0,0
7,smollm2-1.7b,sm-3x3-win,grid,lose,3,1.0,0.0,NaN,0.0,0
8,smollm2-1.7b,sm-3x3-draw,grid,win,3,1.0,0.0,NaN,0.0,0
9,smollm2-1.7b,sm-3x3-draw,grid,lose,3,1.0,0.0,NaN,0.0,0


rows: 18


## 8. Verdict (check mode: fail loudly)

In [14]:
entries = json.loads(STAGE_LOG.read_text()) if STAGE_LOG.exists() else []
fails = [e for e in entries if e["status"] != "ok"]
if fails:
    raise RuntimeError(f"check mode: {len(fails)} failed stages: {[e['stage'] for e in fails]}")
print("ALL CHECK STAGES PASSED")

ALL CHECK STAGES PASSED


## Notes
- **Getting the repo (secret method):** the `GITHUB_TOKEN` secret must be attached to this notebook (+ Add -> Add secret), the notebook SAVED, and the kernel RESTARTED -- env vars are injected at kernel start only.
- **Resume after a died session:** re-run the notebook with a trimmed sweep, e.g. `run_suite.py --models <remaining> --tasks <remaining> --output_dir results/chess`; per-run JSONs under `results/chess/*.summary.json` are the source of truth; the CSV is rebuilt at the end.
- **Gemma models** need the `HF_TOKEN` Kaggle secret (gated access).
- **Timeouts:** full-mode sweep is capped at 12h; typical T4 estimate ~1-2 min/position-cell, well under a single Kaggle session.